In [23]:
import numpy as np
from pyscf import gto, scf, lo, mp, cc

mol = gto.Mole()
mol.verbose = 4
mol.atom = '''
O   -1.485163346097   -0.114724564047    4.000000000000
# H   -1.868415346097    0.762298435953    4.000000000000
H   -0.533833346097    0.040507435953    4.000000000000
O   -1.485163346097   -0.114724564047    0.000000000000
H   -1.868415346097    0.762298435953    0.000000000000
H   -0.533833346097    0.040507435953    0.000000000000
O    1.416468653903    0.111264435953    0.000000000000
H    1.746241653903   -0.373945564047   -0.758561000000
H    1.746241653903   -0.373945564047    0.758561000000
O    1.416468653903    0.111264435953   -4.000000000000
H    1.746241653903   -0.373945564047   -4.758561000000
H    1.746241653903   -0.373945564047   -3.241438999999
'''
mol.spin = 1
mol.basis = 'sto6g'
mol.build()

# xyz_path = "/home/sharmagroup/sharmagroup/project/l7_dataset/geometry/c3a/B.xyz"
# with open(xyz_path) as f:
#     lines = f.readlines()

# second_line = lines[1]
# # charge   = int(second_line.split("charge=")[1].split()[0])
# # spin     = int(second_line.split("mult=")[1].split()[0]) - 1
# # mol_name = second_line.split()[0]
# atoms    = "".join(lines[2:])

# mol = gto.M(
#     atom=atoms,
#     basis="sto6g",
#     verbose=4,
#     unit="angstrom",
#     max_memory=10000,
# )

mf = scf.UHF(mol).density_fit()
mf.kernel()

mymp = mp.MP2(mf).set_frozen()
mymp.kernel()
efull_mp2 = mymp.e_corr
print(f'MP2 Corr = {efull_mp2:.8f}')

mycc = cc.CCSD(mf).set_frozen()
mycc.kernel()
efull_ccsd = mycc.e_corr
print(f'CCSD Corr = {efull_ccsd:.8f}')

efull_t = mycc.ccsd_t()
efull_ccsd_t = efull_ccsd + efull_t
print(f'CCSD(T) Corr = {efull_ccsd_t:.8f}')

System: uname_result(system='Linux', node='yichi-thinkpad', release='4.4.0-26100-Microsoft', version='#8328-Microsoft Fri Jan 01 08:00:00 PST 2016', machine='x86_64')  Threads 12
Python 3.10.16 | packaged by conda-forge | (main, Dec  5 2024, 14:16:10) [GCC 13.3.0]
numpy 1.24.3  scipy 1.14.1  h5py 3.12.1
Date: Sat May 30 00:00:51 2026
PySCF version 2.12.1
PySCF path  /home/yichi/research/software/pyscf
GIT ORIG_HEAD a0665c4a7bf54e33f01295b3eea390be7a17d76d
GIT HEAD (branch master) f97393b29b0a541c155a68d55ee5b652ae7131d2

[ENV] OLD_PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:
[ENV] PYSCF_EXT_PATH /home/yichi/research/software/pyscf-forge:/home/sharmagroup/sharmagroup/pyscf-forge:
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 11
[INPUT] num. electrons = 39
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 1
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = angstrom
[INPUT] Symbol           X                Y                Z      unit     

In [25]:
from afqmc.lno_afqmc import tools
iao_coeff, frag_lolist, atm_center = tools.iao_localization(mf)

In [26]:
def label_elements(elements):
    counters = {}
    labels = []
    
    for element in elements:
        if element not in counters:
            counters[element] = 1
        else:
            counters[element] += 1
        labels.append(f"{element}{counters[element]}")
    
    return labels

labeled_atoms_center = label_elements(atm_center)
print(labeled_atoms_center)

['O1', 'H1', 'O2', 'H2', 'H3', 'O3', 'H4', 'H5', 'O4', 'H6', 'H7']


In [38]:
def plot_density(orbloc, lno_coeff, lno_active, spin_type, idx):
    from pyscf.tools import cubegen
    
    if spin_type == "restricted":
        dm_ctr = orbloc @ orbloc.T
        _ = cubegen.density(mol, f'ctr_density_{idx}.cube', dm_ctr)
        dm_las = lno_coeff[:,lno_active] @ lno_coeff[:,lno_active].T
        _ = cubegen.density(mol, f'las_density_{idx}.cube', dm_las)

    elif spin_type == "unrestricted":
        dm_ctr = orbloc[0] @ orbloc[0].T + orbloc[1] @ orbloc[1].T
        _ = cubegen.density(mol, f'ctr_density_{idx}.cube', dm_ctr)
        dm_las = (lno_coeff[0][:,lno_active[0]] @ lno_coeff[0][:,lno_active[0]].T
                  +lno_coeff[1][:,lno_active[1]] @ lno_coeff[1][:,lno_active[1]].T)
        _ = cubegen.density(mol, f'las_density_{idx}.cube', dm_las)
        
    return None

In [40]:
import numpy as np
from jax import random
from pyscf.lno import lnoccsd
from pyscf.lno import ulnoccsd
from collections.abc import Iterable

from pyscf.data import elements

from afqmc.lno_afqmc import prep
from afqmc.lno_afqmc import mod_lnoccsd

from functools import partial
import time, gc, pickle

# def run_afqmc(mf,
#               lo_coeff = None, 
#               lo_coeff_file = 'lo_coeff.npz',
#               frag_lolist = None,
#               nfrozen = 0,
#               thresh = 1e-6,
#               qmc_options = {}, 
#               chol_cut = 1e-5, 
#               target_sto_error = 1e-3, 
#               run_frg_list = None, 
#               atom_group = None,
#               ):

# mf
lo_coeff = iao_coeff
# lo_coeff_file = 'lo_coeff.npz'.
frag_lolist = frag_lolist
nfrozen = elements.chemcore(mol)
thresh = 1e-4
qmc_options = {}
chol_cut = 1e-5 
target_sto_error = 1e-3
run_frag_list = [0]
atom_group = labeled_atoms_center
plot_las = True

# if lo_coeff is None:
#     try:
#         lo_coeff = np.load(lo_coeff_file)["lo_coeff"]
#     except:
#         raise ValueError(
#             f"lo_coeff was not provided and could not be loaded from '{lo_coeff_file}'")

spin_type = prep.kind(lo_coeff)

if frag_lolist is None:
    if spin_type == "unrestricted":
        raise ValueError("frag_lolist must be provided for unrestricted LNO-AFQMC.")
    print("Fragment list not found. Asign every LO to a fragment.")
    frag_lolist = [[i] for i in range(lo_coeff.shape[1])]

if spin_type == "unrestricted":
    mlno = ulnoccsd.ULNOCCSD(mf, lo_coeff, frag_lolist, frozen=nfrozen).set(verbose=3)
    mf = mlno._scf
else:
    mlno = lnoccsd.LNOCCSD(mf, lo_coeff, frag_lolist, frozen=nfrozen).set(verbose=3)
mlno.lno_thresh = [thresh*10, thresh]
lno_thresh = mlno.lno_thresh
lno_type = ['1h','1h']
eris = mlno.ao2mo()

nfrag = len(frag_lolist)
if run_frag_list is None:
    run_frag_list = range(nfrag)

frag_lolist = [frag_lolist[i] for i in run_frag_list]

lno_pct_occ = [None, None]
lno_norb = [[None,None]] * nfrag

# seeds = random.randint(random.PRNGKey(qmc_options["seed"]),
#                        shape=(nfrag,), 
#                        minval=0, 
#                        maxval=100*nfrag
#                        )

qmc_options["max_error"] = target_sto_error / np.sqrt(nfrag)
trial_base = qmc_options.get("trial", "")

las_center = [None]*nfrag
las_size = np.zeros(nfrag, dtype='int32')
lno_emp2 = np.zeros(nfrag, dtype='float64')
lno_ecc  = np.zeros(nfrag, dtype='float64')
lno_eqmc = np.zeros(nfrag, dtype='float64')
lno_eqmc_err  = np.zeros(nfrag, dtype='float64')
ccsd_time = np.zeros(nfrag, dtype='float64')
qmc_time = np.zeros(nfrag, dtype='float64')

# Loop over fragment
for ifrag, loidx in enumerate(frag_lolist):
    print("\n")
    width = 80
    msg = f" {spin_type} LNO-FRAGMENT {run_frag_list[ifrag]+1}/{nfrag} "
    print(msg.center(width, '='))
    if atom_group is not None:
        atom_msg = f"{atom_group[ifrag]}"
        print(f"Center Atom {atom_msg}")

    if spin_type == "unrestricted":
        orbloc = [lo_coeff[0][:,loidx[0]], lo_coeff[1][:,loidx[1]]]
        lno_param = [
            [
                {
                    'thresh': (
                        lno_thresh[i][s] if isinstance(lno_thresh[i], Iterable)
                        else lno_thresh[i]
                    ),
                    'pct_occ': (
                        lno_pct_occ[i][s] if isinstance(lno_pct_occ[i], Iterable)
                        else lno_pct_occ[i]
                    ),
                    'norb': (
                        lno_norb[ifrag][i][s] if isinstance(lno_norb[ifrag][i], Iterable)
                        else lno_norb[ifrag][i]
                    ),
                } for i in [0, 1]
            ] for s in range(2)
        ]
    else:
        orbloc = lo_coeff[:,loidx]
        lno_param = [{
            'thresh': lno_thresh[i],
            'pct_occ': lno_pct_occ[i],
            'norb': lno_norb[ifrag][i]
            } for i in [0,1]]

    # M = <orbloc|canactocc> (M^dagger M)u = eu
    # u|canactocc> => orbtial in/out the space spanned by |orbloc>
    # uocc_loc = <lno_actocc|orbloc>
    lno_coeff, lno_frozen, uocc_loc, _ = mlno.make_las(eris, orbloc, lno_type, lno_param)
    # lno_coeff still connected to canonical mo_coeff unitarily

    # if spin_type == "unrestricted":
    #     if uocc_loc[0].size > 0 and uocc_loc[1].size == 0:
    #         lno_elec_type = 'alpha'
    #     elif uocc_loc[0].size == 0 and uocc_loc[1].size > 0:
    #         lno_elec_type = 'beta'
    #     else:
    #         lno_elec_type = 'mixed'
    #     print(f'LNO-Frgament Spin Type = {lno_elec_type}')
    #     ao_message_a, ao_max_a = prep.ao_comp(mf, orbloc[0])
    #     ao_message_b, ao_max_b = prep.ao_comp(mf, orbloc[1])
    #     ao_message = ao_message_a + "\n" + ao_message_b
    #     ao_max = ao_max_a + ao_max_b
    # else:
    #     ao_message, ao_max = prep.ao_comp(mf, orbloc)

    mo_occ = mlno.mo_occ

    if spin_type == "unrestricted":
        if uocc_loc[0].size > 0 and uocc_loc[1].size == 0:
            lno_elec_type = 'alpha'
        elif uocc_loc[0].size == 0 and uocc_loc[1].size > 0:
            lno_elec_type = 'beta'
        else:
            lno_elec_type = 'mixed'
        print(f'LNO-Frgament Spin Type = {lno_elec_type}')

        lno_frozen, maskact = ulnoccsd.get_maskact(lno_frozen, [mo_occ[0].size, mo_occ[1].size])
        occidxa = mo_occ[0] > 1e-10
        occidxb = mo_occ[1] > 1e-10
        moidxa, moidxb = maskact
        nactocc_a = int(np.sum(moidxa & occidxa))
        nactvir_a = int(np.sum(moidxa & ~occidxa))
        nactocc_b = int(np.sum(moidxb & occidxb))
        nactvir_b = int(np.sum(moidxb & ~occidxb))
        nactocc = nactocc_a + nactocc_b
        nactvir = nactvir_a + nactvir_b
        print(f'LAS alpha: {nactocc_a} occupied, {nactvir_a} virtual')
        print(f'LAS beta:  {nactocc_b} occupied, {nactvir_b} virtual')
        lno_active_a = np.array([i for i in range(mol.nao) if i not in lno_frozen[0]])
        lno_active_b = np.array([i for i in range(mol.nao) if i not in lno_frozen[1]])
        lno_active = [lno_active_a, lno_active_b]
    else:
        lno_frozen, maskact = lnoccsd.get_maskact(lno_frozen, mo_occ.size)
        lno_active = np.array([i for i in range(mol.nao) if i not in lno_frozen])
        nactocc, nactvir = prep.las_size(mf, lno_frozen)
        print(f'LAS occupied orbitals: {nactocc}')
        print(f'LAS virtual orbitals: {nactvir}')

    if plot_las:
        plot_density(orbloc, lno_coeff, lno_active, spin_type, idx=atom_msg)

    if spin_type == "unrestricted":
        mcc = ulnoccsd.UCCSD(mf, mo_coeff=lno_coeff, frozen=lno_frozen).set(verbose=1)
    else:
        mcc = lnoccsd.CCSD(mf, mo_coeff=lno_coeff, frozen=lno_frozen).set(verbose=1)
    mcc._s1e = mlno._s1e
    mcc._h1e = mlno._h1e
    mcc._vhf = mlno._vhf
    if mlno.kwargs_imp is not None:
        mcc = mcc.set(**mlno.kwargs_imp)
    time0 = time.perf_counter()
    (eorb_mp2, eorb_cc), t1, t2 =\
        mod_lnoccsd.lnoccsd_kernel(mcc, lno_coeff, uocc_loc, mo_occ, maskact)
    time1 = time.perf_counter()
    lnocc_time = time1 - time0

    print(f"CCSD time: {lnocc_time:.6f} s")
    print(f'LNO-MP2 Orbital Energy: {eorb_mp2:.8f}')
    print(f'LNO-CCSD Orbital Energy: {eorb_cc:.8f}')

    if atom_group:
        las_center[ifrag] = atom_msg
    # else:
    #     las_center[ifrag] = ao_max
    las_size[ifrag] = nactocc + nactvir
    lno_emp2[ifrag] = eorb_mp2
    lno_ecc[ifrag] = eorb_cc
    ccsd_time[ifrag] = lnocc_time



======================== unrestricted LNO-FRAGMENT 1/1 =========================
Center Atom O1
LNO-Frgament Spin Type = mixed
LAS alpha: 5 occupied, 1 virtual
LAS beta:  5 occupied, 2 virtual
CCSD time: 0.452570 s
LNO-MP2 Orbital Energy: -0.01071419
LNO-CCSD Orbital Energy: -0.01656580


In [22]:
def plot_density(orbloc, lno_coeff, lno_active, spin_type, idx):
    from pyscf.tools import cubegen
    
    if spin_type == "restricted":
        dm_ctr = orbloc @ orbloc.T
        _ = cubegen.density(mol, f'ctr_density_{idx}.cube', dm_ctr)
        dm_las = lno_coeff[:,lno_active] @ lno_coeff[:,lno_active].T
        _ = cubegen.density(mol, f'las_density_{idx}.cube', dm_las)

    elif spin_type == "unrestricted":
        dm_ctr = orbloc[0] @ orbloc[0].T + orbloc[1] @ orbloc[1].T
        _ = cubegen.density(mol, f'ctr_density_{idx}.cube', dm_ctr)
        dm_las = (lno_coeff[0][:,lno_active] @ lno_coeff[0][:,lno_active].T
                  +lno_coeff[1][:,lno_active] @ lno_coeff[1][:,lno_active].T)
        _ = cubegen.density(mol, f'las_density_{idx}.cube', dm_las)
        
    return None